In [4]:
# Import libraries

import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sqlalchemy import create_engine

In [5]:
from sqlalchemy import create_engine

engine = create_engine(
    "mssql+pyodbc://Hello\\SQLEXPRESS/PortfolioProject_MarketingAnalytics?"
    "driver=ODBC+Driver+18+for+SQL Server&trusted_connection=yes&Encrypt=no"
)

with engine.connect() as conn:
    print("Connected successfully!")


Connected successfully!


C:\Users\surya_t4rpfms\AppData\Local\Temp\ipykernel_7888\3708809013.py:8: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with engine.connect() as conn:


In [6]:
df = pd.read_sql("SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES", engine)
print(df)

         TABLE_NAME
0         customers
1         geography
2          products
3  customer_journey
4  customer_reviews
5   engagement_data


In [7]:
# Step 2 — Load Data

query = "select * from customer_reviews"
df_reviews = pd.read_sql(query,engine)

In [8]:
# Step 3 — Data Validation

df_reviews.sample(5)

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText
1283,1284,2,12,2025-10-04,1,Not worth the money.
1306,1307,36,13,2024-05-16,3,"Average experience, nothing special."
76,77,74,5,2023-03-15,4,"Great purchase, very satisfied."
772,773,17,14,2024-06-21,2,Disappointed with the performance.
64,65,34,3,2024-06-15,1,Not worth the money.


In [9]:
df_reviews.columns = df_reviews.columns.str.strip()

In [10]:
print("NULL counts:\n", df_reviews.isna().sum())
print("Duplicate rows:", df_reviews.duplicated().sum())
print("Zero ratings:", (df_reviews['Rating'] == 0).sum())

NULL counts:
 ReviewID      0
CustomerID    0
ProductID     0
ReviewDate    0
Rating        0
ReviewText    0
dtype: int64
Duplicate rows: 0
Zero ratings: 0


In [11]:
## Step 4 — Sentiment Scoring
analyzer=SentimentIntensityAnalyzer()
df_reviews['SentimentScore']=df_reviews['ReviewText'].apply(lambda x: analyzer.polarity_scores(x)['compound'])

In [12]:
df_reviews.head()

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText,SentimentScore
0,1,77,18,2023-12-23,3,"Average experience, nothing special.",-0.3089
1,2,80,19,2024-12-25,5,The quality is top-notch.,0.0000
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.,0.0000
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper.",0.2382
4,5,64,2,2023-07-16,3,"Average experience, nothing special.",-0.3089


In [13]:
df_reviews[['ReviewText','SentimentScore']].head(10)

,ReviewText,SentimentScore
0,"Average experience, nothing special.",-0.3089
1,The quality is top-notch.,0.0000
2,Five stars for the quick delivery.,0.0000
3,"Good quality, but could be cheaper.",0.2382
4,"Average experience, nothing special.",-0.3089
5,Customer support was very helpful.,0.6997
6,"Average experience, nothing special.",-0.3089
7,The quality is top-notch.,0.0000
8,"I love this product, will buy again!",0.6696
9,"Excellent product, highly recommend!",0.7773


In [14]:
## Step 5 — Sentiment Categorization
def categorize_sentiment(score, rating):
    if score > 0.05: 
        if rating >= 4:
            return 'Positive'  
        elif rating == 3:
            return 'Mixed Positive'  
        else:
            return 'Mixed Negative'  
    elif score < -0.05: 
        if rating <= 2:
            return 'Negative'  
        elif rating == 3:
            return 'Mixed Negative' 
        else:
            return 'Mixed Positive'  
    else:  
        if rating >= 4:
            return 'Positive' 
        elif rating <= 2:
            return 'Negative'  
        else:
            return 'Neutral'

df_reviews['SentimentCategory'] = df_reviews.apply(lambda row: categorize_sentiment(row['SentimentScore'], row['Rating']),axis=1)

In [15]:
df_reviews[['ReviewText', 'Rating', 'SentimentScore', 'SentimentCategory']].head(10)

,ReviewText,Rating,SentimentScore,SentimentCategory
0,"Average experience, nothing special.",3,-0.3089,Mixed Negative
1,The quality is top-notch.,5,0.0000,Positive
2,Five stars for the quick delivery.,4,0.0000,Positive
3,"Good quality, but could be cheaper.",3,0.2382,Mixed Positive
4,"Average experience, nothing special.",3,-0.3089,Mixed Negative
5,Customer support was very helpful.,4,0.6997,Positive
6,"Average experience, nothing special.",3,-0.3089,Mixed Negative
7,The quality is top-notch.,5,0.0000,Positive
8,"I love this product, will buy again!",4,0.6696,Positive
9,"Excellent product, highly recommend!",5,0.7773,Positive


In [16]:
## Step 6 — Sentiment Bucketing
def sentiment_bucket(score):
    if score >= 0.5:
        return '0.5 to 1.0'  
    elif 0.0 <= score < 0.5:
        return '0.0 to 0.49' 
    elif -0.5 <= score < 0.0:
        return '-0.49 to 0.0'  
    else:
        return '-1.0 to -0.5' 
    

df_reviews['SentimentBucket']=df_reviews['SentimentScore'].apply(sentiment_bucket)

In [17]:
df_reviews[['SentimentScore','SentimentBucket']].head(10)

,SentimentScore,SentimentBucket
0,-0.3089,-0.49 to 0.0
1,0.0000,0.0 to 0.49
2,0.0000,0.0 to 0.49
3,0.2382,0.0 to 0.49
4,-0.3089,-0.49 to 0.0
5,0.6997,0.5 to 1.0
6,-0.3089,-0.49 to 0.0
7,0.0000,0.0 to 0.49
8,0.6696,0.5 to 1.0
9,0.7773,0.5 to 1.0


In [18]:
df_reviews.to_csv('customer_reviews_sentiment.csv', index=False)

In [19]:
from collections import Counter
import re
# Filter Negative Reviews
df_negative = df_reviews[df_reviews['SentimentCategory'] == 'Negative']

# Word Frequency Analysis
negative_text = ' '.join(df_negative['ReviewText'].str.lower())
words = re.findall(r'\b\w+\b', negative_text)
word_counts = Counter(words)

stop_words = ['the', 'a', 'is', 'it', 'was', 'and', 'for', 'to', 'of', 'my', 'i', 'not', 'be', 'very', 'this', 'that', 'with', 'on', 'are', 'but']
filtered = {w: c for w, c in word_counts.items() if w not in stop_words}
print(Counter(filtered).most_common(30))

[('product', 112), ('worth', 47), ('money', 47), ('experience', 41), ('did', 38), ('meet', 38), ('expectations', 38), ('arrived', 38), ('late', 38), ('average', 32), ('nothing', 32), ('special', 32), ('disappointed', 22), ('performance', 22), ('after', 18), ('had', 9), ('bad', 9), ('stopped', 9), ('working', 9), ('month', 9), ('okay', 9), ('instructions', 9), ('were', 9), ('unclear', 9), ('terrible', 9), ('customer', 9), ('service', 9), ('would', 9), ('buy', 9), ('again', 9)]


In [20]:
def categorize_issue(text):
    text = text.lower()
    if any(word in text for word in ["worth", "money", "expensive", "price", "costly"]):
        return "Product Cost"
    elif any(word in text for word in ["arrived", "late", "delivery", "shipping"]):
        return "Delivery"
    elif any(word in text for word in ["meet", "expectations", "average", "nothing", "special"]):
        return "Unmet Expectations"
    elif any(word in text for word in ["stopped", "working", "broke", "month"]):
        return "Durability"
    elif any(word in text for word in ["performance", "disappointed", "bad", "terrible"]):
        return "Product Performance"
    elif any(word in text for word in ["instructions", "unclear", "confusing"]):
        return "Usability"
    elif any(word in text for word in ["customer", "service", "support"]):
        return "Customer Service"
    else:
        return "General Dissatisfaction"
    
df_negative = df_reviews[df_reviews['SentimentCategory'] == 'Negative'].copy()

df_negative['IssueCategory'] = df_negative['ReviewText'].apply(categorize_issue)

print(df_negative['IssueCategory'].value_counts())

IssueCategory
Unmet Expectations         70
Product Cost               47
Product Performance        40
Delivery                   38
Durability                 18
Usability                   9
General Dissatisfaction     4
Name: count, dtype: int64


In [21]:
df_negative.to_csv('negative_reviews_issues.csv', index=False)